# Analysis with GeoPandas

## Importing Libraries

In [1]:
 # data analysis library
import pandas as pd
 # geospatial data library
import geopandas as gpd
 # plotting library
import matplotlib.pyplot as plt

## Creating GeoDataFrames

In [2]:
# Creating a GeoDataFrame from coordinate data
data = {
 "City": ["Tokyo", "New York", "London", "Paris"],
 "Latitude": [35.6895, 40.7128, 51.5074, 48.8566],
 "Longitude": [139.6917, -74.0060, -0.1278, 2.3522],
}
# First create a regular pandas DataFrame
df = pd.DataFrame(data)
# Convert to GeoDataFrame by creating Point geometries from coordinates
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.Longitude, df.Latitude)
)
gdf

,City,Latitude,Longitude,geometry
0,Tokyo,35.6895,139.6917,POINT (139.6917 35.6895)
1,New York,40.7128,-74.0060,POINT (-74.006 40.7128)
2,London,51.5074,-0.1278,POINT (-0.1278 51.5074)
3,Paris,48.8566,2.3522,POINT (2.3522 48.8566)


## Reading and Writing

In [4]:
url = "https://github.com/opengeos/datasets/releases/download/vector/nybb.geojson" # GeoJSON file URL
gdf = gpd.read_file(url) # Read the GeoJSON file into a GeoDataFrame
gdf.head() # Display the first few rows of the GeoDataFrame

,BoroCode,BoroName,Shape_Leng,Shape_Area,geometry
0,5,Staten Island,330470.010332,1.623820e+09,"MULTIPOLYGON (((970217.022 145643.332, 970227...."
1,4,Queens,896344.047763,3.045213e+09,"MULTIPOLYGON (((1029606.077 156073.814, 102957..."
2,3,Brooklyn,741080.523166,1.937479e+09,"MULTIPOLYGON (((1021176.479 151374.797, 102100..."
3,1,Manhattan,359299.096471,6.364715e+08,"MULTIPOLYGON (((981219.056 188655.316, 980940...."
4,2,Bronx,464392.991824,1.186925e+09,"MULTIPOLYGON (((1012821.806 229228.265, 101278..."


In [5]:
# Writing GeoDataFrame to a GeoJSON file
output_file = "nyc_boroughs.geojson"
gdf.to_file(output_file, driver="GeoJSON")
print(f"GeoDataFrame has been written to {output_file}")
# Save as Shapefile (traditional GIS format)
output_file = "nyc_boroughs.shp"
gdf.to_file(output_file)
# Save as GeoPackage (modern, single-file format)
output_file = "nyc_boroughs.gpkg"
gdf.to_file(output_file, driver="GPKG")

GeoDataFrame has been written to nyc_boroughs.geojson


c:\Users\arjav\AppData\Local\ESRI\conda\envs\gis222venv\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 1623819823.8099999 of field Shape_Area of feature 0 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\arjav\AppData\Local\ESRI\conda\envs\gis222venv\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 3045212795.1999998 of field Shape_Area of feature 1 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\arjav\AppData\Local\ESRI\conda\envs\gis222venv\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 1937478507.6099999 of field Shape_Area of feature 2 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\arjav\AppData\Local\ESRI\conda\envs\gis222venv\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 636471539.77400005 of field Shape_Area of feature 3 not successfully written.

## Understanding Coordinate Systems

In [6]:
print(f"Current CRS: {gdf.crs}") # Display current CRS

Current CRS: EPSG:2263


In [9]:
# Reproject to WGS84 (latitude/longitude) for global compatibility
gdf_4326 = gdf.to_crs(epsg=4326)
print(f"Reprojected CRS: {gdf_4326.crs}") # Display reprojected CRS
gdf_4326.head() # Display the first few rows of the reprojected GeoDataFrame

Reprojected CRS: EPSG:4326


,BoroCode,BoroName,Shape_Leng,Shape_Area,geometry
0,5,Staten Island,330470.010332,1.623820e+09,"MULTIPOLYGON (((-74.05051 40.56642, -74.05047 ..."
1,4,Queens,896344.047763,3.045213e+09,"MULTIPOLYGON (((-73.83668 40.59495, -73.83678 ..."
2,3,Brooklyn,741080.523166,1.937479e+09,"MULTIPOLYGON (((-73.86706 40.58209, -73.86769 ..."
3,1,Manhattan,359299.096471,6.364715e+08,"MULTIPOLYGON (((-74.01093 40.68449, -74.01193 ..."
4,2,Bronx,464392.991824,1.186925e+09,"MULTIPOLYGON (((-73.89681 40.79581, -73.89694 ..."


In [10]:
 # Number of rows and columns
gdf_4326.shape

(5, 5)

## Spatial Measurements and Analysis

In [11]:
# Reproject to Web Mercator for accurate area calculations in square meters
gdf = gdf.to_crs(epsg=3857)
# Set BoroName as index for easier data access
gdf = gdf.set_index("BoroName")
print(f"Now using CRS: {gdf.crs}")

Now using CRS: EPSG:3857


In [12]:
# Calculate area in square meters
gdf["area"] = gdf.area
# Convert to more readable units (square kilometers)
gdf["area_km2"] = gdf["area"] / 1_000_000
# Display results sorted by area
gdf[["area","area_km2"]].sort_values("area_km2",ascending=False)

,area,area_km2
BoroName,,
Queens,4.928316e+08,492.831621
Brooklyn,3.129694e+08,312.969358
Staten Island,2.618035e+08,261.803516
Bronx,1.929251e+08,192.925108
Manhattan,1.032201e+08,103.220100


## Extracting Geometric Features

In [13]:
# Extract boundary lines from polygons
gdf["boundary"] = gdf.boundary
# Calculate centroids (geometric centers)
gdf["centroid"] = gdf.centroid
# Display the geometric features
gdf[["boundary", "centroid"]].head()

,boundary,centroid
BoroName,,
Staten Island,"MULTILINESTRING ((-8243264.851 4948597.836, -8...",POINT (-8254713.541 4950718.061)
Queens,"MULTILINESTRING ((-8219461.925 4952778.732, -8...",POINT (-8217436.751 4969318.726)
Brooklyn,"MULTILINESTRING ((-8222843.672 4950893.793, -8...",POINT (-8231817.467 4960085.273)
Manhattan,"MULTILINESTRING ((-8238858.864 4965915.024, -8...",POINT (-8233984.803 4979551.765)
Bronx,"MULTILINESTRING ((-8226155.13 4982269.949, -82...",POINT (-8222783.631 4990631.242)


## Distance Calculations

In [17]:
# Use Manhattan's centroid as the reference point
manhattan_centroid = gdf.loc["Manhattan","centroid"]
# Calculate distance from each borough centroid to Manhattan
gdf["distance_to_manhattan"] = gdf.centroid.distance(manhattan_centroid)
# Convert to kilometers and display results
gdf["distance_to_manhattan_km"] = gdf["distance_to_manhattan"] / 1000
gdf[["distance_to_manhattan_km"]].sort_values("distance_to_manhattan_km", ascending=True)

,distance_to_manhattan_km
BoroName,
Manhattan,0.000000
Bronx,15.755034
Queens,19.456442
Brooklyn,19.586772
Staten Island,35.511450


## Summary Statistics

In [15]:
# Calculate summary statistics
mean_distance = gdf["distance_to_manhattan_km"].mean() # Mean distance to Manhattan
max_distance = gdf["distance_to_manhattan_km"].max() # Maximum distance to Manhattan
total_area = gdf["area_km2"].sum()  # Total area of NYC
print(f"Mean distance to Manhattan: {mean_distance:.2f} km")
print(f"Maximum distance to Manhattan: {max_distance:.2f} km")
print(f"Total NYC area: {total_area:.2f} km²")

Mean distance to Manhattan: 18.06 km
Maximum distance to Manhattan: 35.51 km
Total NYC area: 1363.75 km²


In [18]:
gdf['area_km2'].describe() # Summary statistics for area in square kilometers

count      5.000000
mean     272.749941
std      146.018614
min      103.220100
25%      192.925108
50%      261.803516
75%      312.969358
max      492.831621
Name: area_km2, dtype: float64

## Visualizing Geospatial Data

In [ ]:
# Set high resolution for better quality plots


In [ ]:
# Create a choropleth map showing borough areas
fig, ax =  # Set figure size
gdf.plot(
 column= # Color by area in square kilometers
 ax= # Use the specified axes
 legend= # Show legend
 cmap="YlOrRd", # Yellow-Orange-Red colormap
 edgecolor= # Black borders for boroughs
 linewidth= # Border line width
)
 # Title
 # Remove coordinate axes for cleaner appearance
 # Adjust layout for better spacing
 # Display the plot

## Multi-Layer Visualization

In [ ]:
# Create a comprehensive map with multiple layers
fig, ax = # Set figure size
# Plot borough boundaries as base layer

# Add centroids as point layer

# Add borough labels

 # Get centroid coordinates for label placement
 x = # X coordinate
 y = # Y coordinate
 ax.annotate( 
  # Borough label
  # Position at centroid
  # Offset label position
  # Offset in points
  # Font size
  # Bold font
  # Background box
 )
 # Title
 # Remove coordinate axes for cleaner appearance
 # Adjust layout for better spacing
 # Display the plot

## Interactive Visualization

In [ ]:
# Create an interactive map using Folium integration
m =
 column= # Summary statistics for area in square kilometers
 cmap="YlOrRd", # Yellow-Orange-Red colormap
 tooltip= # Tooltip info displayed on hover
 popup= # Popup info on click
 legend= # Display legend
)
m # Display the interactive map

## Advanced Geometric Operations
### Buffer Analysis

In [ ]:
# Create 3-kilometer buffer zones around each borough
buffer_distance = # meters
gdf["buffered"] =  # Create buffers
# Visualize original vs buffered geometries
fig, ax =  # Set figure size
# Plot buffered areas first (background)
gdf
 ax= # use the specified axes
 alpha= # semi-transparent
 color= # Fill color
 edgecolor= # Border color
 linewidth= # Line width
 label= # Label for legend
)
# Plot original geometries on top
gdf
 color=# Fill color
 edgecolor= # Border color
 linewidth= # Line width
 label= # Label for legend
)
 # Title
 # Legend
 # Remove coordinate axes for cleaner appearance
 # Adjust layout for better spacing
 # Display the plot

### Intersection Analysis

In [ ]:
# Test which buffered boroughs intersect with Manhattan's original boundary
manhattan_geom =  # Get Manhattan's geometry
gdf["intersects_manhattan"] =  # Test intersection (Intersection means any overlap)
gdf["touches_manhattan"] =  # Test touching (Touching means they share a boundary but do not overlap)
# Display results
intersection_results =  # Keep only the relevant columns
intersection_results # Display results

# Analysis with PySAL

In [ ]:
import  # geospatial data library
   # Display available example datasets

In [ ]:

guerry =  # Load Guerry dataset
# guerry dataset contains historical socio-economic and moral statistics for French départements

In [ ]:
from libpysal.examples import  # Load explain function    
    # Display information about the Guerry dataset


In [ ]:
data= # Get path to the shapefile
gdf =  # Read the shapefile into a GeoDataFrame
gdf

In [ ]:
 # Import Queen contiguity class
y =  # Extract 'Donatns' column as values (Donations per capita)
w =  # Create Queen contiguity weights
 # Recode weights to row-standardized form

In [ ]:
 # Import Moran's I class
moran =  # Create Moran's I object
 # Moran's I statistic 
 # p-value
 # z-score

In [ ]:
 # Import Moran's I plotting function
 # Plot Moran's I
plt.show() # Display the plot